# Lab 4: Parameter-efficient fine-tuning

---
## Statement of AI tool use
During this lab, we mostly use ChatGPT for terms explanation and code samples about usage of python libraries. We also used codex-cli help us review and find issues. All the code, answers were written by us. 

---

## Feedback and Revisions

All assignments completed in Lab 4: Yes.

Overall comment: Please mark your solutions with a clear header so they are easy to find.


Task 4.07:

We requested a review about our implementation because we thought the result is unexpected (accuracy is always higher no matter the rank).

Feedback:

In this assignment, you do not have to create a new class and approximate. You can use the approximate you already made. The replace returns the model, so you should return replace here.

The suggested solution uses the headtuned and finetuned (not the pretrained really). I am not 100% sure if it works or not to use you class solution as I did not run the code,

but I would suggest just doing dela as headtuned_layers[name].weight.data - finetuned_layers[name].weight.data with torch.no_grad() while looping over or name, layer in headtuned_layers.items().
Then doing headtuned + approximate(delta, rank) and copy that result to a clone (clone_linear(layer)) in the loop. We got rank 3.

Amends:

We simplified the implementation in 4.06 and 4.07 and reran all related code but still get the same result. It's still better than head-tuned model given r=1.



---

Fine-tuning all parameters of pre-trained language models can be resource-intensive. Because of this, current research in natural language processing is looking into developing methods for adapting models to downstream tasks without full fine-tuning. These methods only tune a small number of model parameters while yielding performance comparable to that of a fully fine-tuned model.

In this lab, you will implement LoRA, one of the most well-known methods for parameter-efficient fine-tuning. LoRA stands for “Low-Rank Adaptation of Large Language Models” and was originally described in a research article by [Hu et al. (2021)](https://arxiv.org/abs/2106.09685).

Along the way, you will earn experience with [Hugging Face Transformers](https://huggingface.co/docs/transformers/en/index), a state-of-the-art library for training and deploying language models, as well as with several related libraries. In particular, you will learn a best-practice workflow for downloading a Transformer model and fine-tuning it on the downstream task of binary sentiment classification.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Dataset

The data for this lab comes from the [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/). The full dataset consists of 50,000 highly polar movie reviews collected from the Internet Movie Database (IMDB). Here, we use a random sample consisting of 2,000 reviews for training and 500 reviews for evaluation.

To load the dataset, we use the [Hugging Face Datasets](https://huggingface.co/docs/datasets/en/index) library.

In [1]:
from datasets import load_dataset

imdb_dataset = load_dataset(
    "csv", data_files={"train": "train.csv", "eval": "eval.csv"}
)

imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 500
    })
})

As we can see, each sample in the dataset is a record with three fields: an internal index (`index`, an integer), the text of the review (`review`, a string), and the sentiment label (`label`, an integer – 1&nbsp;for “positive” and 0&nbsp;for “negative” sentiment).

Here is an example record:

In [2]:
imdb_dataset["train"][645]

{'index': 2981,
 'review': 'Brilliant execution in displaying once and for all, this time in the venue of politics, of how "good intentions do actually pave the road to hell". Excellent!',
 'label': 1}

## Tokeniser

As our pre-trained language model, we will use [DistilBERT](https://huggingface.co/docs/transformers/en/model_doc/distilbert), a compact encoder model with 40% less parameters than BERT base. DistilBERT is not actually a *large* language model by modern standards and thus does not benefit as much from parameter-efficient fine-tuning as other models. However, it has the benefit of being light and fast, and can be run even on consumer hardware.

To feed the movie reviews to DistilBERT, we need to tokenise them and encode the resulting tokens as integers in the model vocabulary. We start by loading the DistilBERT tokeniser using the [Auto classes](https://huggingface.co/docs/transformers/en/model_doc/auto):

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

We then create a tokenised version of the dataset:

In [4]:
def tokenize_function(batch):
    return tokenizer(batch["review"], padding=True, truncation=True)


tokenized_imdb_dataset = imdb_dataset.map(tokenize_function, batched=True)

tokenized_imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
})

As we can see, tokenising adds two additional fields to each review: `input_ids` is the list of token ids corresponding to the review, and `attention_mask` is the list of indices specifying which tokens the encoder should attend to.

To avoid trouble when fine-tuning the model later, the next cell disables tokeniser parallelism.

In [5]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Trainer

In this section, we will set up our workflow for training and evaluating DistilBERT models. The central component in this workflow is the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), which provides extensive configuration options. Here, we leave most of these options at their default value. Two changes we *do* make are to enable evaluation of the trained model after each epoch, and to log the training and evaluation loss after every 5&nbsp;training steps (the default is 500).

In [40]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="tmp_trainer",
    eval_strategy="epoch",
    seed=42,
    logging_steps=5,
)

In addition to the loss, we also track classification accuracy. For this we import the [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/en/index) library and define a small helper function `compute_metrics()` that the trainer will call after each epoch.

In [41]:
import evaluate

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In the next cell we define a convenience function `make_trainer()` that creates a readily-configured trainer for a specified model (*model*). We will use this trainer both to train the model on the training section of the tokenised review dataset, and to evaluate it on the evaluation section.

In [42]:
from transformers import Trainer
from transformers.utils.notebook import NotebookProgressCallback

def make_trainer(model):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_imdb_dataset["train"],
        eval_dataset=tokenized_imdb_dataset["eval"],
        compute_metrics=compute_metrics,
    )
    trainer.remove_callback(NotebookProgressCallback)
    return trainer

## Full fine-tuning

In the rest of this notebook, we will work our way to the implementation of LoRA, and compare LoRA to traditional fine-tuning methods. Our first point of reference is a fully fine-tuned DistilBERT model.

We start by loading the pre-trained model:

In [9]:
from transformers import AutoModelForSequenceClassification

pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

pretrained_model

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


The architecture of DistilBERT is that of a standard Transformer encoder with an embedding layer (`embeddings`) followed by a stack of six Transformer blocks (`transformer`) and a feedforward network with two linear layers (`pre_classifier` and `classifier`) and a final dropout layer (`dropout`).

### 🧩 Task 4.01: Counting the number of trainable parameters

One relevant measure in the context of parameter-efficient fine-tuning is the number of parameters that need to be changed when training a model. Your first task in this lab is to write a function `num_trainable_parameters()` that calculates this number for a given model.

In [10]:
def num_trainable_parameters(model):
    # TODO: Replace the next line with your own code
    return sum((p.numel() for p in model.parameters() if p.requires_grad))

In [11]:
num_trainable_parameters(pretrained_model)

66955010

The function should implement the following specification:

> **num_trainable_parameters** (*model*)
>
> Returns the number of float-valued trainable parameters in the specified *model* as an integer.

#### 👍 Hint

The term *parameter* can refer to either complete tensors or the individual elements of these tensors. For example, a linear layer created by `nn.Linear(3, 5)` has 2&nbsp;tensor-valued parameters (a weight matrix and a bias vector) and 20&nbsp;float-valued parameters (the elements of these tensors). To get the tensor-valued parameters of a model, you can use the [`parameters()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters) method. A parameter is *trainable* if it requires gradient.

#### 🤞 Test your code

To test your code, apply your function to the pre-trained model. The correct number of float-valued trainable parameters for this model is 66,955,010.

### Fine-tuning

When we load the pre-trained model, the Hugging Face Transformers library warns us that the weights of the feedforward network have not yet been trained. To do so, in the next cell, we pass the pre-trained model to a trainer and initiate the fine-tuning process.

**⚠️ Please note that fine-tuning the model will take some time! ⚠️**

You can work on the other problems in this lab while you are waiting.

In [12]:
finetuned_trainer = make_trainer(pretrained_model)

finetuned_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.536857,0.416414,0.866000
2,0.004964,0.405822,0.910000
3,0.002099,0.418463,0.914000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.23767545926570893, metrics={'train_runtime': 158.3993, 'train_samples_per_second': 37.879, 'train_steps_per_second': 4.735, 'total_flos': 794804391936000.0, 'train_loss': 0.23767545926570893, 'epoch': 3.0})

Because full fine-tuning is so resource-intensive, we save the fine-tuned model to disk:

In [13]:
finetuned_trainer.save_model("finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Later in this notebook, whenever you need the fully fine-tuned version of the model, you can load it as follows:

In [14]:
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### Convenience functions

Because we will repeat the steps we just took to fine-tune the pre-trained model several times in this notebook, we define two convenience functions:

In [45]:
def train(model):
    print("Number of trainable parameters:", num_trainable_parameters(model))
    trainer = make_trainer(model)
    trainer.train()
    return model

In [46]:
def evaluate(model):
    trainer = make_trainer(model)
    return trainer.evaluate()

In [17]:
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")
evaluate(finetuned_model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.4184630811214447,
 'eval_model_preparation_time': 0.001,
 'eval_accuracy': 0.914,
 'eval_runtime': 3.9968,
 'eval_samples_per_second': 125.1,
 'eval_steps_per_second': 15.763}

## Tuning the final layers only

If full fine-tuning marks one end of the complexity spectrum, the other end is marked by only tuning the final layers of the transformer – the *head* of the model. In the case of DistilBERT, the head consists of the `pre_classifier` and `classifier` layers.

### 🧩 Task 4.02: Head-tuning

Implement the head-tuning strategy by coding the following function:

In [13]:
def make_headtuned_model():
    # TODO: Replace the next line with your own code
#       (pre_classifier): Linear(in_features=768, out_features=768, bias=True)
#   (classifier): Linear(in_features=768, out_features=2, bias=True)
#   (dropout): Dropout(p=0.2, inplace=False)
# )
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    for name, p in model.named_parameters():
        if 'distilbert' in name:
            p.requires_grad = False
    return model

another_model = make_headtuned_model()

another_model

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


Here is the specification of this function:

> **make_headtuned_model** ()
>
> Returns a model that is identical to the pre-trained model, except that the head layers have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.)

#### 👍 Hint

You freeze a parameter by setting its `requires_grad`-attribute to `False`.

Once you have an implementation of the head-tuning strategy, evaluate it on the evaluation data. How much accuracy do we lose when only training the final layers of the pre-trained model, compared to full fine-tuning?

---
#### Answer 4.02
We lose 10pp accuracy (80.6% vs 90.4%)

---

In [43]:
headtuned_model = make_headtuned_model()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
num_trainable_parameters(headtuned_model)

592130

#### 🤞 Test your code

If you configured your model correctly, `num_trainable_parameters()` should show 592,130 trainable parameters.

For future reference, we also save the head-tuned model:

In [47]:
headtuned_model = make_headtuned_model()
train(headtuned_model)
evaluate(headtuned_model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of trainable parameters: 592130


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.5140107274055481,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.806,
 'eval_runtime': 4.0075,
 'eval_samples_per_second': 124.767,
 'eval_steps_per_second': 15.721,
 'epoch': 0}

In [48]:
make_trainer(headtuned_model).save_model("headtuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Layer surgery

LoRA works by “wrapping” frozen layers from the pre-trained Transformer model inside adapter modules. Conventionally, this wrapping is only applied to the linear layers that transform the queries and values in the self-attention mechanism. To implement the wrapping, we need functions to extract and replace layers in a model. Your task in this section is to code these functions.

### 🎓 Task 4.03: Extracting layers

Code a function that extracts the query and value linear layers from a DistilBERT model:

In [17]:
def extract(model):
    q_keys = [f'distilbert.transformer.layer.{i}.attention.q_lin' for i in range(6)]
    v_keys =  [f'distilbert.transformer.layer.{i}.attention.v_lin' for i in range(6)]
    qs = {q_key: model.get_submodule(q_key) for q_key in q_keys}
    vs = {v_key: model.get_submodule(v_key) for v_key in v_keys}
    
    return {**qs, **vs}

In [18]:
d = extract(pretrained_model)

In [19]:
sum([num_trainable_parameters(d[k]) for k in d])

7087104

Implement this function to match the following specification:

> **extract** (*model*)
>
> Takes a DistilBERT model (*model*) and extracts the query and value linear layers from each block of the Transformer. Returns a dictionary mapping the DistilBERT module names of these layers to the layers themselves (instances of `nn.Linear`).

#### 👍 Hint

As we saw earlier, the DistilBERT model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. Use [`get_submodule()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.get_submodule) to retrieve a layer by name. You can hard-wire the names of the layers you want to extract.

#### 🤞 Test your code

To test your code, check the number of trainable float-valued parameters in the extracted layers. This number should be 7,087,104.

### 🎓 Task 4.04: Replacing layers

Next, code the inverse of the `extract()` function to replace selected layers of a module using a dictionary of named layers.

In [20]:
def replace(model, named_layers):
    # TODO: Replace the next line with your own code
    for key, layer in named_layers.items():
        module = '.'.join(key.split('.')[:-1])
        suffix = key.split('.')[-1]
        setattr(model.get_submodule(module), suffix, layer)
        
    return model

Implement this function to match the following specification:

> **replace** (*model*, *named_layers*)
>
> Takes a DistilBERT model (*model*) and a dictionary in the format returned by `extract()` (*named_layers*) and injects the extracted layers into the model. More specifically, suppose that *named_layers* contains a key–value pair `(name, layer)`. Then the function replaces the submodule of *model* addressed by the fully-qualified string name `name` by the layer `layer`. Returns the modified model.

#### 👍 Hint

Use [`getattr()`](https://docs.python.org/3/library/functions.html#getattr) and [`setattr()`](https://docs.python.org/3/library/functions.html#setattr) to return or set the value of a named submodule.

#### 🤞 Test your code

To test your implementation, write code that (1)&nbsp;extracts the query and value linear layers from the fine-tuned model; (2)&nbsp;replaces these layers with clones with random weights; and (3)&nbsp;replaces these layers again with the original versions. Evaluating the modified model after step&nbsp;(2) should yield a near-random accuracy. Evaluating it again after step&nbsp;(3) should yield the original accuracy.

The following function should be helpful. It clones a linear layer, copying the weights and the bias from the original.

In [21]:
import torch.nn as nn


def clone_linear(original):
    out_features, in_features = original.weight.shape
    copy = nn.Linear(in_features, out_features)
    copy.load_state_dict(original.state_dict())
    return copy

In [22]:
import torch

def randomize(origin):
    l = clone_linear(origin)
    with torch.no_grad():
        l.weight = torch.nn.Parameter(torch.rand(l.weight.shape))
    return l

In [30]:
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

named_layers = extract(finetuned_model)

cloned_random_layers = {}

for k in named_layers:
    cloned_random_layers[k] = randomize(named_layers[k])

replace(finetuned_model, cloned_random_layers)

evaluate(finetuned_model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.7019376158714294,
 'eval_model_preparation_time': 0.001,
 'eval_accuracy': 0.526,
 'eval_runtime': 4.0138,
 'eval_samples_per_second': 124.569,
 'eval_steps_per_second': 15.696}

In [31]:
replace(finetuned_model, named_layers)

evaluate(finetuned_model)

{'eval_loss': 0.4184630811214447,
 'eval_model_preparation_time': 0.001,
 'eval_accuracy': 0.914,
 'eval_runtime': 4.0638,
 'eval_samples_per_second': 123.038,
 'eval_steps_per_second': 15.503}

---

After replaced with random weight, its accuracy decreased to 52.6%. And after replaced with origin cloned weight, its accuracy turn back into 90.4%

## Low-rank approximation

The basic idea behind LoRA is to conceptualise fine-tuned weights as a sum $W_0 + \Delta W$ of the weights from the pre-trained model, $W_0$, and a low-rank update matrix $\Delta W$. The goal of fine-tuning, then, is to learn the update matrix; this happens in the adapter layers.

Before we get to the implementation of the LoRA adapter layers, we first check to what extent the assumption that fine-tuning can be described by low-rank matrices holds true for DistilBERT. To do so, we will “cheat” and replace the query and value linear layers of the head-tuned model with low-rank approximations. The technical key to this is the truncated singular value decomposition (SVD).

### 🎓 Task 4.05: Low-rank matrix approximation

Your first task in this section is to implement the low-rank matrix approximation.

In [16]:
def approximate(matrix, rank):
    # TODO: Replace the next line with your own code
    U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)
    U_r = U[:, :rank]
    S_r = S[:rank]
    Vh_r = Vh[:rank, :]
    return U_r @ torch.diag(S_r) @ Vh_r

Implement this function to match the following specification:

> **approximate** (*matrix*, *rank*)
>
> Takes a 2D-tensor (*matrix*) and an integer rank $r$ (*rank*), computes the truncated SVD with rank $r$ on the tensor, and returns the corresponding low-rank approximation matrix.

#### 👍 Hint

If you need a refresher on the low-rank matrix approximation, read the corresponding section from the Wikipedia article on the [Singular value decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition#Low-rank_matrix_approximation). The truncated SVD is an extension of the full SVD; the latter can be computed using [`torch.linalg.svd()`](https://pytorch.org/docs/stable/generated/torch.linalg.svd.html).

#### 🤞 Test your code

To test your code, run the following cell. It creates a matrix `original` with rank $r \leq 8$ and after that the rank-$8$ approximation matrix `approximation`. You should find that the distance between the two matrices is very low.

In [33]:
original = torch.rand(768, 8) @ torch.rand(8, 384)
approximation = approximate(original, 8)
torch.dist(original, approximation)

tensor(0.0003)

### 🎓 Task 4.06: Approximated fine-tuned model (version 1)

In the next step, your task is to construct a version of the head-tuned model in which every query and value linear layer is replaced by a low-rank approximation of the corresponding layer from the fully fine-tuned model.

In [49]:
def approximate(linear, r):
    new_layer = clone_linear(linear)
    with torch.no_grad():
        W = new_layer.weight.data
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        U_r, S_r, Vh_r = U[:, :r], S[:r], Vh[:r, :]

        new_layer.weight.copy_(U_r @ torch.diag(S_r) @ Vh_r)
        
        return new_layer

def make_approximated_model_1(rank):
    # TODO: Replace the next line with your own code
    
    headtuned_model = AutoModelForSequenceClassification.from_pretrained("headtuned")
    finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")
    
    named_layers = extract(finetuned_model)

    lora = {}
    
    for k in named_layers:
        lora[k] = approximate(named_layers[k], rank)
    
    replace(headtuned_model, lora)
    return headtuned_model

Here is the specification of this function:

> **make_approximated_model_1** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which every query and value linear layer is replaced by its $r$-approximated corresponding layer from the fully fine-tuned model.

Run the next cell to evaluate your model for different rank values. Start with the full rank and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

---
#### Answer 4.06
It's 384. When the rank turned to 192 the accuracy is already 76.8% (vs 80.6% from head-tuned model).

---

In [50]:
approximated_model_1 = make_approximated_model_1(768)

evaluate(approximated_model_1)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.36380627751350403,
 'eval_model_preparation_time': 0.0009,
 'eval_accuracy': 0.878,
 'eval_runtime': 3.8287,
 'eval_samples_per_second': 130.591,
 'eval_steps_per_second': 16.455,
 'epoch': 0}

In [51]:
approximated_model_1 = make_approximated_model_1(384)

evaluate(approximated_model_1)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.4221702814102173,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.834,
 'eval_runtime': 3.8525,
 'eval_samples_per_second': 129.785,
 'eval_steps_per_second': 16.353,
 'epoch': 0}

In [52]:
approximated_model_1 = make_approximated_model_1(192)

evaluate(approximated_model_1)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.49626678228378296,
 'eval_model_preparation_time': 0.0013,
 'eval_accuracy': 0.768,
 'eval_runtime': 3.8857,
 'eval_samples_per_second': 128.678,
 'eval_steps_per_second': 16.213,
 'epoch': 0}

### 🎓 Task 4.07: Approximated fine-tuned model (version 2)

In the approximated model from the previous section, the truncated SVD is applied to the full weight matrix of the fine-tuned model: $W_0 + \Delta W$. In LoRA, the low-rank approximation only applies to the *update matrix* $\Delta W$, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

In [53]:
def approximate2(f_linear, p_linear, r):
    base = clone_linear(p_linear)
    linear = clone_linear(f_linear)
    with torch.no_grad():
        dW = linear.weight.data - base.weight.data
        
        delta_layer = clone_linear(p_linear) # a temporary layer with dW to calculate low rank approximation of dW
        delta_layer.weight.data.copy_(dW)
        
        approx_delta = approximate(delta_layer, r) # get a linear layer with low rank approximation of dW
        
        new_layer = clone_linear(p_linear)
        new_layer.weight.data.copy_(p_linear.weight.data + approx_delta.weight.data)

        return new_layer

def make_approximated_model_2(rank):
    # TODO: Replace the next line with your own code
    finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")
    headtuned_model = AutoModelForSequenceClassification.from_pretrained("headtuned")

    
    finetuned_layers = extract(finetuned_model)
    pretrained_layers = extract(headtuned_model)

    lora = {}
    
    for k in finetuned_layers:
        lora[k] = approximate2(finetuned_layers[k], pretrained_layers[k], rank)
    
    return replace(headtuned_model, lora)


Implement the function to match the following specification:

> **make_approximated_model_2** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which the weight matrix of every query and value linear layer is replaced by the sum $W_0 + \Delta W$, where $W_0$ is the weight matrix of the pre-trained model and $\Delta W$ is the rank-$r$ approximation of the update matrix, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

Run the next cell to evaluate your model for different rank values. Start with the rank from the approximated model from the previous section and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

---
#### Answer 4.07:
It's always better than head-tuned model given $r>0$. Even when `r=1`, its accuracy is 85% (vs 80.6% from headtuned model).

---

In [54]:
approximated_model_2 = make_approximated_model_2(384)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.36504149436950684,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.878,
 'eval_runtime': 3.8472,
 'eval_samples_per_second': 129.966,
 'eval_steps_per_second': 16.376,
 'epoch': 0}

In [55]:
approximated_model_2 = make_approximated_model_2(192)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.36534520983695984,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.878,
 'eval_runtime': 3.8669,
 'eval_samples_per_second': 129.303,
 'eval_steps_per_second': 16.292,
 'epoch': 0}

In [56]:
approximated_model_2 = make_approximated_model_2(96)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.3657018542289734,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.878,
 'eval_runtime': 3.8934,
 'eval_samples_per_second': 128.423,
 'eval_steps_per_second': 16.181,
 'epoch': 0}

In [57]:
approximated_model_2 = make_approximated_model_2(48)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.36675670742988586,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.88,
 'eval_runtime': 3.9051,
 'eval_samples_per_second': 128.038,
 'eval_steps_per_second': 16.133,
 'epoch': 0}

In [58]:
approximated_model_2 = make_approximated_model_2(24)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.36901983618736267,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.88,
 'eval_runtime': 3.9207,
 'eval_samples_per_second': 127.527,
 'eval_steps_per_second': 16.068,
 'epoch': 0}

In [59]:
approximated_model_2 = make_approximated_model_2(12)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.3737475574016571,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.878,
 'eval_runtime': 3.9351,
 'eval_samples_per_second': 127.062,
 'eval_steps_per_second': 16.01,
 'epoch': 0}

In [60]:
approximated_model_2 = make_approximated_model_2(6)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.3810323476791382,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.868,
 'eval_runtime': 3.9285,
 'eval_samples_per_second': 127.274,
 'eval_steps_per_second': 16.036,
 'epoch': 0}

In [61]:
approximated_model_2 = make_approximated_model_2(3)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.38713306188583374,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.872,
 'eval_runtime': 3.9577,
 'eval_samples_per_second': 126.336,
 'eval_steps_per_second': 15.918,
 'epoch': 0}

In [62]:
approximated_model_2 = make_approximated_model_2(1)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.4491710066795349,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.854,
 'eval_runtime': 3.9542,
 'eval_samples_per_second': 126.448,
 'eval_steps_per_second': 15.932,
 'epoch': 0}

In [63]:
approximated_model_2 = make_approximated_model_2(0)

evaluate(approximated_model_2)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.5140107274055481,
 'eval_model_preparation_time': 0.0008,
 'eval_accuracy': 0.806,
 'eval_runtime': 3.9583,
 'eval_samples_per_second': 126.317,
 'eval_steps_per_second': 15.916,
 'epoch': 0}

## Low-Rank Adaptation (LoRA)

In this section, you will implement the LoRA adapters and fine-tune the adapted model.

### 🎓 Task 4.08: Implement the adapter

A LoRA adapter implements the forward function

$$
y = x W_0 + x \Delta W = x W_0 + x A B
$$

where $W_0$ is a linear transformation from the pre-trained model and $\Delta W$ is a learned update matrix, deconstructed into the product $AB$ of two rank-$r$ matrices $A$ and $B$. LoRA scales the update matrix $\Delta W$ by a factor of $\alpha / r$, where $\alpha$ is a hyperparameter. (To keep the formula tidy, we ignore the fact that the linear transformation in the pre-trained model may additionally include a bias.)

In [57]:
import torch.nn as nn
import torch.nn.functional as F


class LoRA(nn.Module):
    def __init__(self, pretrained, rank=12, alpha=24):
        super().__init__()
        # TODO: Add your code here
        self.base = pretrained
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.r, self.alpha = rank, alpha
        self.scaling = alpha / rank
        self.A = nn.Parameter(torch.randn(rank, pretrained.in_features))
        self.B = nn.Parameter(torch.zeros(pretrained.out_features, rank))
        # nn.init.kaiming_uniform_(self.A, a=5**0.5)
        # nn.init.zeros_(self.B)

    def forward(self, x):
        # TODO: Replace the next line with your own code
        y = self.base(x)
        lora = F.linear(F.linear(x, self.A), self.B) * self.scaling
        return y + lora

Your code must comply with the following specification:

**__init__** (*self*, *pretrained*, *rank* = 12, *alpha* = 24)

> Initialises the LoRA adapter. This sets up the matrices $A$ and $B$ from the equation above. The matrix $A$ is initialised with random weights from a standard normal distribution; the matrix $B$ is initialised with zeros. The argument *pretrained* is the linear layer from the pre-trained model that should be adapted. The arguments *rank* and *alpha* are the rank $r$ and the hyperparameter $\alpha$ in the equation above.

**forward** (*self*, *x*)

> Sends an input *x* through the adapter, implementing the equation above.

### 🎓 Task 4.09: Inject the adapter into the pre-trained model

The final step is to construct an adapted model by injecting the LoRA adapters into the pre-trained model.

In [62]:
def make_lora_model(rank):
    # TODO: Replace the next line with your own code
    pretrained_model = pretrained_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    
    for p in pretrained_model.distilbert.parameters():
        p.requires_grad = False
    
    named_layers = extract(pretrained_model)

    lora = {}
    
    for k in named_layers:
        lora[k] = LoRA(named_layers[k], rank)
    
    replace(pretrained_model, lora)
    train(pretrained_model)
    return pretrained_model

Implement the function to match the following specification:

> **make_lora_model** (*rank*)
>
> Returns a model that is identical to the pre-trained model, except that the query and value linear layers have been wrapped in LoRA adapters, and the LoRA adapters and the head layers of the pre-trained model have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.) The rank of the adapters is specified by the argument *rank*. The *alpha* value of the adapters is set to twice the rank (a common rule of thumb).

Run the next cell to evaluate your model for $r = 6$ and $\alpha = 12$. How many trainable parameters does the adapted model have? What accuracy do you get? How do these value relate to the number of trainable parameters and accuracy of the fully fine-tuned model, in terms of percentages?

In [63]:
lora_model = make_lora_model(6)

evaluate(lora_model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of trainable parameters: 702722


Epoch,Training Loss,Validation Loss,Accuracy
1,0.477635,0.312059,0.876000
2,0.155702,0.340256,0.898000
3,0.147654,0.411908,0.886000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.41190841794013977,
 'eval_model_preparation_time': 0.0011,
 'eval_accuracy': 0.886,
 'eval_runtime': 4.121,
 'eval_samples_per_second': 121.331,
 'eval_steps_per_second': 15.288}

---
#### Answer 4.09
Trainable Parameters: ~0.7m

Accuracy: 88.6%

Fully finetune model's trainable parameters: ~66m

Fully finetune model's accuracy: 91.4%

LoRA model (r=6)'s size is 1% of fully finetuned one and achieved 96.9% of fully finetuned's accuracy

---

## Alternatives to Transformer-based models

Even with methods for parameter-efficient fine-tuning, applying DistilBERT and other Transformer-based models comes at a significant cost – an investment that does not always pay off. In the final task of this lab, we ask you to explore a more traditional approach to classification and contrast it with the pre-training/fine-tuning approach of neural language models.

### 🎓 Task 4.10: Comparing with a non-neural classifier

Browse the web to find a tutorial on how to apply a classifier from the [scikit-learn](https://scikit-learn.org/stable/) library to the problem of sentiment classification and implement the method here in this notebook. We suggest you use [multinomial Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) or [logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). (Once you have code for one method, it is easy to switch to the other.) Evaluate your chosen classifier on the IMDB dataset.

Questions to consider:

* Which classifier did you try? What results did you get? How long did it take you to train and run the classifier?
* What is your perspective on the trade-off between accuracy and resource requirements between the two approaches?
* What did you learn? How, exactly, did you learn it? Why does this learning matter?

---
#### Answer 4.10:
- We used nultinomial Naive Bayes with TF-IDF tokenizer. We get 81.8% accuracy on evaluation set. It takes 0.18s to train and 0.04s to evaluate.
- The non-neural classifier is obviously worse in accuracy (81.8% vs 90%). But it trains and evaluates much faster and is less depending on resources.
- I learned that a traditional non-neural classifier can perform well on sentiment-classification tasks while neural classifier could do better with more computing resources. I learned it through comparing the run time and accuracy of two approaches. It helps a lot when deciding which approach to apply for a real world task. Sometimes a classical approach just works and a advanced method is not necessary because it's resource-intense.
---

In [54]:
imdb_dataset = load_dataset(
    "csv", data_files={"train": "train.csv", "eval": "eval.csv"}
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import numpy as np
import time

x_train, y_train = imdb_dataset['train']['review'], np.array(imdb_dataset['train']['label'])
x_eval, y_eval = imdb_dataset['eval']['review'], np.array(imdb_dataset['eval']['label'])
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10_000, stop_words="english")),
    ("nb", MultinomialNB(alpha=0.1))
])

begin_t = time.time()

pipe.fit(x_train, y_train)

print(f'train used: {time.time() - begin_t} s')
begin_t = time.time()

y_pred = pipe.predict(x_eval)

print(f'eval used: {time.time() - begin_t} s')

acc = accuracy_score(y_eval, y_pred)

print(acc)

train used: 0.19301390647888184 s
eval used: 0.04535055160522461 s
0.818


**🥳 Congratulations on finishing this lab! 🥳**